# SOEN 471 - SmartCart Assignment Notebook

## Part 1 - Data PreProcessing

In [4]:
# Load data
import pandas as pd

user_data = pd.read_csv('data/ecommerce_user_data.csv')
product_data = pd.read_csv('data/product_details.csv')

print(user_data.head())
print(product_data.head())

  UserID ProductID  Rating   Timestamp  Category
0   U000     P0009       5  2024-09-08     Books
1   U000     P0020       1  2024-09-02      Home
2   U000     P0012       4  2024-10-18     Books
3   U000     P0013       1  2024-09-18  Clothing
4   U000     P0070       4  2024-09-16      Toys
  ProductID      ProductName     Category
0     P0000      Toys Item 0     Clothing
1     P0001  Clothing Item 1  Electronics
2     P0002     Books Item 2  Electronics
3     P0003  Clothing Item 3  Electronics
4     P0004  Clothing Item 4  Electronics


In [5]:
# Clean the data (its already pretty clean)

# --- user_data ---

# 1. Fix data types
user_data['Timestamp'] = pd.to_datetime(user_data['Timestamp'], errors='coerce')

# 2. Standardize strings
user_data['UserID']    = user_data['UserID'].str.strip()
user_data['ProductID'] = user_data['ProductID'].str.strip()
user_data['Category']  = user_data['Category'].str.strip().str.lower()

# 3. Standardize string columns
user_data['Category'] = user_data['Category'].str.strip().str.lower()
user_data['UserID']  = user_data['UserID'].astype(str).str.strip()

# 4. Validate rating range (should be 1–5)
invalid_ratings = user_data[~user_data['Rating'].between(1, 5)]
print(f"Invalid ratings: {len(invalid_ratings)}")
user_data = user_data[user_data['Rating'].between(1, 5)]

# 5. Drop duplicates (same user rating same product twice)
before = len(user_data)
user_data = user_data.drop_duplicates(subset=['UserID', 'ProductID'])
print(f"Duplicate user-product pairs removed: {before - len(user_data)}")

# --- product_data ---

# 1. Standardize strings
product_data['ProductID']   = product_data['ProductID'].str.strip()
product_data['ProductName'] = product_data['ProductName'].str.strip()
product_data['Category']    = product_data['Category'].str.strip().str.lower()

# 3. Drop duplicate products
before = len(product_data)
product_data = product_data.drop_duplicates(subset=['ProductID'])
print(f"Duplicate products removed: {before - len(product_data)}")

# --- Summary ---
print(f"\nuser_data shape:    {user_data.shape}")
print(f"product_data shape: {product_data.shape}")
print("\nuser_data dtypes:\n",    user_data.dtypes)
print("\nproduct_data dtypes:\n", product_data.dtypes)

Invalid ratings: 0
Duplicate user-product pairs removed: 0
Duplicate products removed: 0

user_data shape:    (724, 5)
product_data shape: (100, 3)

user_data dtypes:
 UserID                  str
ProductID               str
Rating                int64
Timestamp    datetime64[us]
Category                str
dtype: object

product_data dtypes:
 ProductID      str
ProductName    str
Category       str
dtype: object


In [6]:
# Convert into user-item matrix
user_item_matrix = user_data.pivot_table(
        values='Rating',
        index='UserID',
        columns='ProductID',
        fill_value=0
    )

print(user_item_matrix.head())

ProductID  P0000  P0001  P0002  P0003  P0004  P0005  P0006  P0007  P0008  \
UserID                                                                     
U000         0.0    0.0    0.0    3.0    0.0    5.0    0.0    3.0    0.0   
U001         0.0    0.0    3.0    0.0    0.0    0.0    0.0    0.0    0.0   
U002         0.0    0.0    0.0    0.0    0.0    5.0    0.0    0.0    0.0   
U003         0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
U004         0.0    3.0    0.0    0.0    0.0    0.0    2.0    0.0    0.0   

ProductID  P0009  ...  P0090  P0091  P0092  P0093  P0094  P0095  P0096  P0097  \
UserID            ...                                                           
U000         5.0  ...    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
U001         0.0  ...    0.0    5.0    0.0    0.0    0.0    3.0    0.0    0.0   
U002         0.0  ...    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
U003         0.0  ...    0.0    0.0    0.0    0.0    0.0    0.

In [7]:
# Group by user and category
agg_user_data = user_data.groupby(['UserID', 'Category'])['Rating'].agg(
    AverageRating='mean',
    Count='count'
)

print(agg_user_data.head(20))

                    AverageRating  Count
UserID Category                         
U000   books             3.666667      6
       clothing          1.666667      3
       electronics       3.666667      3
       home              1.000000      2
       toys              3.500000      6
U001   beauty            4.000000      1
       books             2.500000      4
       clothing          2.500000      4
       electronics       4.000000      2
       home              2.000000      1
       toys              4.000000      1
U002   beauty            3.000000      3
       books             1.000000      1
       clothing          1.000000      1
       electronics       2.000000      1
       home              2.000000      2
       toys              2.833333      6
U003   beauty            3.333333      3
       books             3.000000      1
       clothing          1.333333      3
